# Training Pipeline with dHT

This notebook demonstrates how to set up a training pipeline for models using dHT tokenization.

## Training Strategies

There are several ways to train with dHT:

1. **End-to-end training**: Train tokenizer and transformer together
2. **Two-stage training**: Pre-train tokenizer, then train transformer
3. **Fine-tuning**: Adapt pre-trained models with dHT

## Key Considerations

- Variable number of tokens per image
- Attention masking for batching
- Gradient flow through tokenizer
- Memory management with adaptive tokens

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from dht.nn.transformer import dHTClassifier
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Create Training Data

For demonstration, we'll create synthetic data.

In [ ]:
# Create synthetic dataset
def create_dummy_dataset(n_samples=100, n_classes=10):
    images = torch.randn(n_samples, 3, 224, 224)
    labels = torch.randint(0, n_classes, (n_samples,))
    return TensorDataset(images, labels)

# Create datasets
train_dataset = create_dummy_dataset(100, n_classes=10)
val_dataset = create_dummy_dataset(20, n_classes=10)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

## 2. Initialize Model

In [ ]:
# Create model
model = dHTClassifier(
    embed_dim=384,
    patch_size=16,
    heads=6,
    depth=6,  # Smaller for demo
    n_classes=10,
    channels=3,
    compute_grad=True,
    dop_path=0.1,  # Stochastic depth
).to(device)

# Count parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params/1e6:.2f}M")

## 3. Set Up Optimizer and Loss

In [ ]:
# Optimizer
optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=0.05
)

# Loss function
criterion = nn.CrossEntropyLoss()

# Learning rate scheduler
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=10,
    eta_min=1e-6
)

print("Training setup complete")

## 4. Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Track metrics
        total_loss += loss.item()
        _, predicted = logits.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return total_loss / len(loader), 100. * correct / total


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            _, predicted = logits.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    return total_loss / len(loader), 100. * correct / total

In [ ]:
# Training loop
num_epochs = 5
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    scheduler.step()
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
    print(f"Epoch {epoch+1}/{num_epochs}:")
    print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

## 5. Plot Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss
ax1.plot(train_losses, label='Train')
ax1.plot(val_losses, label='Val')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.legend()
ax1.grid(True)

# Accuracy
ax2.plot(train_accs, label='Train')
ax2.plot(val_accs, label='Val')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## 6. Advanced: Freezing Components

You can freeze parts of the model for efficient fine-tuning:

In [ ]:
# Freeze tokenizer
for param in model.tokenizer.parameters():
    param.requires_grad = False

# Or freeze transformer blocks
model.freeze_blocks()

# Or unfreeze
model.unfreeze_blocks()

# Count trainable parameters
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {n_trainable/1e6:.2f}M / {n_total/1e6:.2f}M parameters")

## 7. Saving and Loading Models

In [ ]:
# Save model
torch.save({
    'epoch': num_epochs,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_loss': train_losses[-1],
    'val_loss': val_losses[-1],
}, '/tmp/dht_checkpoint.pth')

print("Model saved to /tmp/dht_checkpoint.pth")

# Load model
checkpoint = torch.load('/tmp/dht_checkpoint.pth')
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

print(f"Model loaded from epoch {checkpoint['epoch']}")

## Summary

This notebook covered:
1. Setting up datasets and dataloaders
2. Initializing dHT models
3. Training loops with proper gradient flow
4. Validation and metrics tracking
5. Freezing/unfreezing components
6. Saving and loading checkpoints

### Key Points:
- dHT works with standard PyTorch training loops
- Variable token counts are handled automatically
- Attention masks ensure proper batching
- Can freeze tokenizer or transformer independently

### Next: Embedding Fix (Notebook 04)